# QRAP Grover's Algorithm Benchmark

This notebook demonstrates and benchmarks Grover's quantum search algorithm simulation using the QRAP library.

Grover's algorithm provides quadratic speedup for unstructured search problems:
- Classical search: O(N) queries
- Quantum search: O(√N) queries

In [ ]:
# Import required libraries
try:
    import qrap_python as qrap
except ImportError:
    print("QRAP Python bindings not installed. Run 'maturin develop' in the python/ directory.")
    qrap = None

import matplotlib.pyplot as plt
import numpy as np
import time

## 1. Basic Grover Simulation

Let's start with a simple Grover search:

In [ ]:
if qrap:
    # Create a Grover simulator with 4 qubits (16 states)
    simulator = qrap.GroverSimulator(num_qubits=4)
    
    print(f"Number of qubits: {simulator.num_qubits}")
    print(f"Search space size: {simulator.search_space_size}")
    print(f"Optimal iterations: {simulator.optimal_iterations()}")

In [ ]:
if qrap:
    # Search for a target value
    target = 7
    result = simulator.run(target)
    
    print(f"Target: {result.target}")
    print(f"Found: {result.found}")
    print(f"Success: {result.success}")
    print(f"Iterations: {result.iterations}")
    print(f"Probability: {result.probability:.4f}")

## 2. Probability Distribution Visualization

In [ ]:
if qrap:
    # Get probability distribution before and after running
    simulator = qrap.GroverSimulator(num_qubits=4)
    initial_probs = simulator.get_probabilities()
    
    # Run the algorithm
    target = 5
    result = simulator.run(target)
    final_probs = simulator.get_probabilities()
    
    # Plot
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Initial distribution
    states = range(len(initial_probs))
    colors = ['red' if i == target else 'blue' for i in states]
    
    axes[0].bar(states, initial_probs, color=colors, alpha=0.7)
    axes[0].set_xlabel('State')
    axes[0].set_ylabel('Probability')
    axes[0].set_title('Initial Probability Distribution')
    axes[0].set_ylim(0, 1)
    
    # Final distribution
    axes[1].bar(states, final_probs, color=colors, alpha=0.7)
    axes[1].set_xlabel('State')
    axes[1].set_ylabel('Probability')
    axes[1].set_title(f'After {result.iterations} Grover Iterations (target={target})')
    axes[1].set_ylim(0, 1)
    
    plt.tight_layout()
    plt.savefig('grover_probability_distribution.png', dpi=150)
    plt.show()

## 3. Benchmark: Success Rate vs Number of Qubits

In [ ]:
if qrap:
    # Run benchmarks for different qubit counts
    qubit_counts = [2, 3, 4, 5, 6, 7, 8]
    num_runs = 100
    success_rates = []
    
    for n_qubits in qubit_counts:
        results = qrap.run_benchmark(n_qubits, num_runs)
        successes = sum(1 for r in results if r.success)
        rate = successes / num_runs
        success_rates.append(rate)
        print(f"{n_qubits} qubits: {rate:.2%} success rate")
    
    # Plot
    plt.figure(figsize=(10, 6))
    plt.plot(qubit_counts, success_rates, 'bo-', linewidth=2, markersize=8)
    plt.axhline(y=0.5, color='r', linestyle='--', label='Random guess')
    plt.xlabel('Number of Qubits')
    plt.ylabel('Success Rate')
    plt.title(f'Grover Algorithm Success Rate ({num_runs} runs per configuration)')
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.ylim(0, 1.1)
    plt.savefig('grover_success_rate.png', dpi=150)
    plt.show()

## 4. Benchmark: Iterations Analysis

In [ ]:
if qrap:
    # Analyze success rate vs iterations for 4 qubits
    simulator = qrap.GroverSimulator(num_qubits=4)
    target = 7
    max_iterations = 10
    num_trials = 100
    
    iteration_success = []
    
    for iters in range(1, max_iterations + 1):
        successes = 0
        for _ in range(num_trials):
            result = simulator.run_with_iterations(target, iters)
            if result.success:
                successes += 1
        iteration_success.append(successes / num_trials)
    
    # Plot
    plt.figure(figsize=(10, 6))
    plt.bar(range(1, max_iterations + 1), iteration_success, alpha=0.7)
    optimal = simulator.optimal_iterations()
    plt.axvline(x=optimal, color='r', linestyle='--', label=f'Optimal iterations ({optimal})')
    plt.xlabel('Number of Iterations')
    plt.ylabel('Success Rate')
    plt.title('Success Rate vs Number of Grover Iterations (4 qubits, target=7)')
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.savefig('grover_iterations_analysis.png', dpi=150)
    plt.show()

## 5. Performance Timing

In [ ]:
if qrap:
    # Time the simulation for different qubit counts
    qubit_counts = [4, 6, 8, 10, 12]
    times = []
    
    for n_qubits in qubit_counts:
        simulator = qrap.GroverSimulator(num_qubits=n_qubits)
        
        start = time.time()
        for _ in range(10):  # 10 runs for averaging
            simulator.run(target=1)
        elapsed = (time.time() - start) / 10
        
        times.append(elapsed * 1000)  # Convert to ms
        print(f"{n_qubits} qubits: {elapsed*1000:.3f} ms per run")
    
    # Plot
    plt.figure(figsize=(10, 6))
    plt.semilogy(qubit_counts, times, 'go-', linewidth=2, markersize=8)
    plt.xlabel('Number of Qubits')
    plt.ylabel('Time per Run (ms)')
    plt.title('Grover Simulation Performance')
    plt.grid(True, alpha=0.3)
    plt.savefig('grover_performance.png', dpi=150)
    plt.show()

## 6. Summary Statistics

In [ ]:
if qrap:
    # Generate comprehensive benchmark report
    print("=" * 60)
    print("QRAP Grover Benchmark Summary")
    print("=" * 60)
    
    for n_qubits in [4, 6, 8]:
        results = qrap.run_benchmark(n_qubits, 200)
        
        successes = sum(1 for r in results if r.success)
        probs = [r.probability for r in results]
        
        print(f"\n{n_qubits} Qubits (2^{n_qubits} = {2**n_qubits} states):")
        print(f"  Success rate: {successes/len(results):.2%}")
        print(f"  Avg probability: {np.mean(probs):.4f}")
        print(f"  Min probability: {np.min(probs):.4f}")
        print(f"  Max probability: {np.max(probs):.4f}")
        print(f"  Optimal iterations: {results[0].iterations}")